In [ ]:
import sys
from io import StringIO

old_stderr = sys.stderr
sys.stderr = StringIO()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LearningSparkV2") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

sys.stderr = old_stderr


# Spark Tables

This notebook shows how to use Spark Catalog Interface API to query databases, tables, and columns.

A full list of documented methods is available [here](https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.Catalog)

In [ ]:
us_flights_file = "/Users/abhikashyap10/spark/LearningSparkV2/databricks-datasets/learning-spark-v2/flights/departuredelays.csv"

### Create Managed Tables

In [ ]:
# Create database and managed tables
spark.sql("DROP DATABASE IF EXISTS learn_spark_db CASCADE") 
spark.sql("CREATE DATABASE learn_spark_db")
spark.sql("USE learn_spark_db")
spark.sql("CREATE TABLE us_delay_flights_tbl(date STRING, delay INT, distance INT, origin STRING, destination STRING)")

### Display the databases

In [ ]:
display(spark.catalog.listDatabases())

## Read our US Flights table

In [ ]:
df = (spark.read.format("csv")
      .schema("date STRING, delay INT, distance INT, origin STRING, destination STRING")
      .option("header", "true")
      .option("path", "/Users/abhikashyap10/spark/LearningSparkV2/databricks-datasets/learning-spark-v2/flights/departuredelays.csv")
      .load())

## Save into our table

In [ ]:
df.write.mode("overwrite").saveAsTable("us_delay_flights_tbl")

## Cache the Table

In [ ]:
# SQL cell
CACHE TABLE us_delay_flights_tbl

Check if the table is cached

In [ ]:
spark.catalog.isCached("us_delay_flights_tbl")

### Display tables within a Database

Note that the table is MANGED by Spark

In [ ]:
spark.catalog.listTables(dbName="learn_spark_db")

### Display Columns for a table

In [ ]:
spark.catalog.listColumns("us_delay_flights_tbl")

### Create Unmanaged Tables

In [ ]:
# Drop the database and create unmanaged tables
spark.sql("DROP DATABASE IF EXISTS learn_spark_db CASCADE")
spark.sql("CREATE DATABASE learn_spark_db")
spark.sql("USE learn_spark_db")
spark.sql("CREATE TABLE us_delay_flights_tbl (date STRING, delay INT, distance INT, origin STRING, destination STRING) USING csv OPTIONS (path '/Users/abhikashyap10/spark/LearningSparkV2/databricks-datasets/learning-spark-v2/flights/departuredelays.csv')")

### Display Tables

**Note**: The table type here that tableType='EXTERNAL', which indicates it's unmanaged by Spark, whereas above the tableType='MANAGED'

In [ ]:
spark.catalog.listTables(dbName="learn_spark_db")

### Display Columns for a table

In [ ]:
spark.catalog.listColumns("us_delay_flights_tbl")